# Notebook 12: Final Project – Autonomous Assistant

**🎯 Goal:** Combine everything we've learned—RAG, Tools, Memory, and LangGraph—to build a powerful, autonomous AI assistant that can answer questions about documents, search the web, and hold a conversation.

## 🧩 Concept Overview

This is the capstone project! We're building a single, unified agent that can intelligently decide which tool to use to answer a user's question. It will be able to:

1.  **Perform RAG:** Answer questions about a specific document (`annual-report.pdf`) by retrieving relevant information from a vectorstore.
2.  **Search the Web:** Use a search tool to find up-to-date information on any topic.
3.  **Remember the Conversation:** Use memory to handle follow-up questions and maintain context.
4.  **Reason and Act:** Use LangGraph to manage the state and decide the best course of action (e.g., "Should I use the RAG tool, the web search tool, or just respond directly?").

## 🖼️ Visual Diagram: Final Agent Architecture

```
                                +--------------------+
                                |   Memory Buffer    |
                                +---------+----------+
                                          ^      |
                                          |      | (Injects History)
                                          |      v
User Query ---> +----------------------------------------------------+
                | LangGraph Agent (State: messages, ...)             | ---> Final Answer
                |                                                    |
                |  +------------------+     +----------------------+ |
                |  | Supervisor LLM   | --> | Tool Executor        | |
                |  | (Decides action) |     |                      | |
                |  +------------------+     +----------+-----------+ |
                |        |                           |             |
                |        | (Route to...)             | (Calls...)  |
                |        v                           v             |
                |  +-----------------+     +-------------------+   |
                |  | RAG Tool        |     | Web Search Tool   |   |
                |  | (for doc Q's)   |     | (for general Q's) |   |
                |  +-----------------+     +-------------------+   |
                +----------------------------------------------------+
```

## ⚙️ Setup

We'll be importing many components we've used throughout the series.

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Sequence
import operator

# LLM
from langchain_openai import ChatOpenAI

# RAG Components
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Agent Components
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, AIMessage
from langgraph.prebuilt import ToolExecutor

load_dotenv()
llm = ChatOpenAI(model="gpt-4o", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. Component 1: The RAG Tool

First, we'll build our RAG pipeline and then wrap it in a `@tool` so our agent can use it.

In [ ]:
# Load and process the document
loader = TextLoader('../data/annual-report.pdf')
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

# Create the RAG chain
rag_prompt = ChatPromptTemplate.from_template(
    "Answer the user's question based only on the following context:\n\n{context}\n\nQuestion: {input}"
)
rag_chain = create_stuff_documents_chain(llm, rag_prompt)
retrieval_chain = create_retrieval_chain(retriever, rag_chain)

# Wrap the RAG chain in a tool
@tool
def query_annual_report(query: str) -> str:
    """Use this tool to answer questions about the Innovate Inc. 2024 annual report. 
    It can provide information on financial highlights, projects like Chimera and Terra, and future outlook."""
    response = retrieval_chain.invoke({"input": query})
    return response['answer']

print("RAG tool created successfully.")
# Test the tool
# print(query_annual_report.invoke("What were the total revenues?"))

## 2. Component 2: The Web Search Tool

Next, we'll add a general-purpose web search tool.

In [ ]:
@tool
def web_search(query: str) -> str:
    """A general-purpose web search tool for finding up-to-date information on any topic."""
    search = DuckDuckGoSearchRun()
    return search.run(query)

print("Web search tool created successfully.")

## 3. Component 3: The LangGraph Agent

Now we'll assemble our agent using LangGraph, giving it access to the two tools we just created.

In [ ]:
tools = [query_annual_report, web_search]
tool_executor = ToolExecutor(tools)
llm_with_tools = llm.bind_tools(tools)

# Define the agent state
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# Define the nodes
def call_model(state):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def call_tool(state):
    last_message = state["messages"][-1]
    tool_outputs = []
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        print(f"-- Calling tool: {tool_name} --")
        output = tool_executor.invoke(tool_call)
        tool_outputs.append(ToolMessage(content=str(output), tool_call_id=tool_call['id']))
    return {"messages": tool_outputs}

# Define the conditional edge
def should_continue(state):
    if not isinstance(state["messages"][-1], AIMessage):
        return "end"
    return "continue" if state["messages"][-1].tool_calls else "end"

# Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("action", call_tool)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue, {"continue": "action", "end": END})
workflow.add_edge('action', 'agent')

assistant_app = workflow.compile()
print("Autonomous Assistant is ready!")

## 4. Running the Assistant

Let's test our assistant's capabilities. We'll start a conversation and ask it different types of questions.

In [ ]:
conversation_history = []

def run_assistant(question):
    global conversation_history
    
    # Add the new question to the history
    conversation_history.append(HumanMessage(content=question))
    
    # Invoke the app
    result = assistant_app.invoke({"messages": conversation_history})
    
    # Update history with the AI's response
    conversation_history = result['messages']
    
    # Print the final response
    print(f"\nAssistant: {conversation_history[-1].content}")

# --- Let's Chat! ---
print("--- Test 1: RAG Question ---")
run_assistant("What is Project Chimera from the annual report?")

print("\n--- Test 2: Web Search Question ---")
run_assistant("What's the latest news about NASA's Artemis program?")

print("\n--- Test 3: Follow-up Question (Memory) ---")
run_assistant("Can you summarize the main goal of that program?")

## 🔧 Final Exercise: Add a New Tool

**Goal:** Enhance the assistant by giving it a new capability: a calculator.

1.  Define a simple `calculator` tool that takes a string expression and returns the result.
2.  Create a new list of tools that includes the `query_annual_report`, `web_search`, AND your new `calculator` tool.
3.  Re-compile the `assistant_app` with the new tool list.
4.  Ask the assistant a question that requires both RAG and calculation: `"What were the total revenues in the annual report, and what is that number divided by 12?"`

In [ ]:
# 1. Define the calculator tool
@tool
def calculator(expression: str) -> float:
    """Evaluates a mathematical expression."""
    return eval(expression)

# 2. Create the new tool list
final_tools = [query_annual_report, web_search, calculator]
final_tool_executor = ToolExecutor(final_tools)
final_llm_with_tools = llm.bind_tools(final_tools)

# 3. Re-build the graph components with the new tools
def final_call_model(state):
    return {"messages": [final_llm_with_tools.invoke(state["messages"])]}

def final_call_tool(state):
    last_message = state["messages"][-1]
    tool_outputs = []
    for tool_call in last_message.tool_calls:
        output = final_tool_executor.invoke(tool_call)
        tool_outputs.append(ToolMessage(content=str(output), tool_call_id=tool_call['id']))
    return {"messages": tool_outputs}

final_workflow = StateGraph(AgentState)
final_workflow.add_node("agent", final_call_model)
final_workflow.add_node("action", final_call_tool)
final_workflow.set_entry_point("agent")
final_workflow.add_conditional_edges("agent", should_continue, {"continue": "action", "end": END})
final_workflow.add_edge('action', 'agent')
final_app = final_workflow.compile()

# 4. Run the new query
final_conversation_history = []
question = "What were the total revenues in the annual report, and what is that number divided by 12?"
final_conversation_history.append(HumanMessage(content=question))
result = final_app.invoke({"messages": final_conversation_history})
print(f"Assistant: {result['messages'][-1].content}")

## 🚀 Next Steps & Deployment

This agent is now a powerful, runnable application. The next step would be to deploy it so others can use it.

**LangServe** is the recommended way to deploy LangChain applications. It automatically creates a REST API endpoint for your compiled LangGraph app.

```python
# In a file named `server.py`
from fastapi import FastAPI
from langserve import add_routes

# Assuming 'final_app' is your compiled LangGraph app
app = FastAPI(title="My Assistant Server")
add_routes(
    app,
    final_app,
    path="/assistant",
)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
```
You could then build a simple frontend using a framework like Streamlit or Next.js that interacts with this API.

## 🎉 Series Summary

Congratulations on completing the LangChain Learning Series! You have journeyed from the basics of LLMs to building a sophisticated, autonomous assistant.

You now have the skills to:
- **Communicate with LLMs** using prompts and parsers.
- **Build chains** for sequential tasks.
- **Perform RAG** to ground LLMs in your private data.
- **Create intelligent agents** that can reason, use tools, and remember conversations.
- **Leverage advanced architectures** like LangGraph for complex, stateful workflows.
- **Debug and evaluate** your applications with LangSmith.

The world of LLM-powered application development is just beginning. Keep experimenting, keep building, and see what amazing things you can create!